# WP contestation and boundary redrawing analysis

Investigates whether Workers' Party contestation is associated with subsequent boundary changes.

In [3]:
# Cell 1: Setup and data loading
import pandas as pd
import geopandas as gpd
import numpy as np
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

# Style
sns.set_style('whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 150

# Load election results
elections = pd.read_csv('../../raw_data/demographics/election_results_raw.csv')
# Convert numeric columns
elections['vote_percentage'] = pd.to_numeric(elections['vote_percentage'], errors='coerce')
elections['vote_count'] = pd.to_numeric(elections['vote_count'], errors='coerce')
print(f'Election results: {len(elections)} rows, years {elections.year.min()}-{elections.year.max()}')

# Filter to WP, 2006+
wp = elections[(elections.party == 'WP') & (elections.year >= 2006)].copy()
wp['constituency'] = wp['constituency'].str.strip().str.upper()
print(f'WP rows (2006+): {len(wp)}')
print(f'vote_percentage dtype: {wp.vote_percentage.dtype}')

# Load boundary changes GeoJSON
boundary_gdf = gpd.read_file('../../boundary_changes_final.geojson')
print(f'Boundary cells: {len(boundary_gdf)}')
print(f'Properties: {list(boundary_gdf.columns)}')

# Parse h field if needed (already a list in GeoJSON)
if isinstance(boundary_gdf.iloc[0]['h'], str):
    boundary_gdf['h'] = boundary_gdf['h'].apply(json.loads)
print(f'Sample h: {boundary_gdf.iloc[0]["h"]}')

Election results: 1609 rows, years 1955-2025
WP rows (2006+): 39
vote_percentage dtype: float64
Boundary cells: 1762
Properties: ['n', 'c', 'h', 'geometry']
Sample h: ['ALJUNIED' 'ANG MO KIO' 'ALJUNIED' 'ANG MO KIO' 'ALJUNIED']


In [4]:
# Cell 2: Build WP contestation lookup

years = [2006, 2011, 2015, 2020, 2025]
wp_contested = {}  # year -> set of UPPERCASE constituency names
wp_results = {}    # year -> {constituency: {vote_pct, candidates}}

for year in years:
    yr_data = wp[wp.year == year]
    wp_contested[year] = set(yr_data.constituency.unique())
    wp_results[year] = {}
    for _, row in yr_data.iterrows():
        wp_results[year][row.constituency] = {
            'vote_pct': round(row.vote_percentage * 100, 2) if pd.notna(row.vote_percentage) else None,
            'candidates': row.candidates
        }

print('WP contests per year:')
for year in years:
    names = sorted(wp_contested[year])
    print(f'  {year} ({len(names)}): {names}')

WP contests per year:
  2006 (7): ['ALJUNIED', 'ANG MO KIO', 'EAST COAST', 'HOUGANG', 'JOO CHIAT', 'NEE SOON CENTRAL', 'NEE SOON EAST']
  2011 (8): ['ALJUNIED', 'EAST COAST', 'HOUGANG', 'JOO CHIAT', 'MOULMEIN-KALLANG', 'NEE SOON', 'PUNGGOL EAST', 'SENGKANG WEST']
  2015 (10): ['ALJUNIED', 'EAST COAST', 'FENGSHAN', 'HOUGANG', 'JALAN BESAR', 'MACPHERSON', 'MARINE PARADE', 'NEE SOON', 'PUNGGOL EAST', 'SENGKANG WEST']
  2020 (6): ['ALJUNIED', 'EAST COAST', 'HOUGANG', 'MARINE PARADE', 'PUNGGOL WEST', 'SENGKANG']
  2025 (8): ['ALJUNIED', 'EAST COAST', 'HOUGANG', 'JALAN KAYU', 'PUNGGOL', 'SENGKANG', 'TAMPINES', 'TAMPINES CHANGKAT']


In [5]:
# Cell 3: Compute area churned per constituency per transition

# Compute area for each cell (in projected CRS for accurate area)
boundary_proj = boundary_gdf.to_crs(epsg=3414)  # SVY21 - Singapore projected CRS
boundary_gdf['area_m2'] = boundary_proj.geometry.area

# GRC expansions that should NOT count as boundary changes:
# the underlying constituency identity was preserved, only the GRC name changed
GRC_EXPANSION_EXCLUSIONS = {
    ('WEST COAST', 'WEST COAST-JURONG WEST'),
    ('MARINE PARADE', 'MARINE PARADE-BRADDELL HEIGHTS'),
}

def is_genuine_change(old_name, new_name):
    if old_name == new_name:
        return False
    return (old_name, new_name) not in GRC_EXPANSION_EXCLUSIONS

transitions = [
    (2006, 2011, 0, 1),
    (2011, 2015, 1, 2),
    (2015, 2020, 2, 3),
    (2020, 2025, 3, 4),
]

rows = []
for year_t, year_t1, idx_t, idx_t1 in transitions:
    # Group cells by constituency name at time t
    boundary_gdf['const_t'] = boundary_gdf['h'].apply(lambda h: h[idx_t])
    boundary_gdf['const_t1'] = boundary_gdf['h'].apply(lambda h: h[idx_t1])
    boundary_gdf['changed'] = boundary_gdf.apply(
        lambda row: is_genuine_change(row['const_t'], row['const_t1']), axis=1
    )
    
    grouped = boundary_gdf.groupby('const_t').agg(
        total_area=('area_m2', 'sum'),
        churned_area=('area_m2', lambda x: x[boundary_gdf.loc[x.index, 'changed']].sum()),
        n_cells=('area_m2', 'count')
    ).reset_index()
    
    grouped['churn_rate'] = grouped['churned_area'] / grouped['total_area']
    grouped['wp_contested'] = grouped['const_t'].isin(wp_contested[year_t])
    grouped['transition'] = f'{year_t}→{year_t1}'
    grouped['year'] = year_t
    
    # Attach WP vote percentage
    grouped['wp_vote_pct'] = grouped['const_t'].apply(
        lambda c: wp_results[year_t].get(c, {}).get('vote_pct', None)
    )
    
    grouped.rename(columns={'const_t': 'constituency'}, inplace=True)
    rows.append(grouped)

churn_df = pd.concat(rows, ignore_index=True)
print(f'Total constituency-transition observations: {len(churn_df)}')
print(f'WP-contested: {churn_df.wp_contested.sum()}, Non-WP: {(~churn_df.wp_contested).sum()}')
print()
print(churn_df.groupby('transition').agg(
    n_constituencies=('constituency', 'count'),
    n_wp=('wp_contested', 'sum'),
    mean_churn=('churn_rate', 'mean'),
    mean_churn_wp=('churn_rate', lambda x: x[churn_df.loc[x.index, 'wp_contested']].mean()),
    mean_churn_nonwp=('churn_rate', lambda x: x[~churn_df.loc[x.index, 'wp_contested']].mean()),
).to_string())
print()
print('Sample rows:')
churn_df[churn_df.wp_contested].head(10)

Total constituency-transition observations: 110
WP-contested: 31, Non-WP: 79

            n_constituencies  n_wp  mean_churn  mean_churn_wp  mean_churn_nonwp
transition                                                                     
2006→2011                 23     7    0.349626       0.363048          0.343753
2011→2015                 27     8    0.168270       0.299810          0.112885
2015→2020                 29    10    0.159725       0.329837          0.070193
2020→2025                 31     6    0.305451       0.366476          0.290805

Sample rows:


,constituency,total_area,churned_area,n_cells,churn_rate,wp_contested,transition,year,wp_vote_pct
0,ALJUNIED,2.947299e+07,1.066021e+06,143,0.036169,True,2006→2011,2006,43.91
1,ANG MO KIO,4.665980e+07,1.154370e+07,155,0.247401,True,2006→2011,2006,33.86
5,EAST COAST,1.125567e+08,1.100564e+06,79,0.009778,True,2006→2011,2006,36.14
8,HOUGANG,1.488905e+06,3.888465e+04,35,0.026116,True,2006→2011,2006,62.74
10,JOO CHIAT,4.806844e+06,1.066494e+06,19,0.221870,True,2006→2011,2006,34.99
14,NEE SOON CENTRAL,1.223929e+06,1.223929e+06,1,1.000000,True,2006→2011,2006,34.63
15,NEE SOON EAST,5.802589e+06,5.802589e+06,4,1.000000,True,2006→2011,2006,31.28
23,ALJUNIED,2.922877e+07,5.795670e+04,145,0.001983,True,2011→2015,2011,54.72
28,EAST COAST,1.114610e+08,2.511360e+06,89,0.022531,True,2011→2015,2011,45.17
31,HOUGANG,1.450923e+06,1.548542e+04,31,0.010673,True,2011→2015,2011,64.80


In [6]:
# Cell 4: Charts

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Chart 1: Strip/swarm plot
ax1 = axes[0, 0]
palette = {True: '#08519c', False: '#bdbdbd'}
sns.stripplot(
    data=churn_df, x='wp_contested', y='churn_rate',
    hue='transition', dodge=True, jitter=0.25, alpha=0.6,
    ax=ax1, size=6
)
# Overlay means with CI
for i, contested in enumerate([False, True]):
    vals = churn_df[churn_df.wp_contested == contested]['churn_rate']
    mean = vals.mean()
    ci = 1.96 * vals.std() / np.sqrt(len(vals))
    ax1.errorbar(i, mean, yerr=ci, fmt='D', color='black', markersize=8, capsize=6, zorder=10)
ax1.set_xlabel('WP Contested')
ax1.set_ylabel('Churn Rate (area fraction redrawn)')
ax1.set_title('Boundary Churn: WP vs Non-WP Constituencies')
ax1.set_xticklabels(['No', 'Yes'])

# Chart 2: WP contestation timeline heatmap
ax2 = axes[0, 1]
all_consts = sorted(set().union(*wp_contested.values()))
heatmap_data = pd.DataFrame(0.0, index=all_consts, columns=years)
for year in years:
    for const in wp_contested[year]:
        pct = wp_results[year].get(const, {}).get('vote_pct', 0)
        heatmap_data.loc[const, year] = pct if pct else 0
# Sort by total WP presence
heatmap_data['count'] = (heatmap_data > 0).sum(axis=1)
heatmap_data = heatmap_data.sort_values('count', ascending=True)
heatmap_data.drop('count', axis=1, inplace=True)
sns.heatmap(
    heatmap_data, annot=True, fmt='.1f', cmap='Blues',
    linewidths=0.5, ax=ax2, vmin=0, vmax=65,
    cbar_kws={'label': 'WP Vote Share (%)'}
)
ax2.set_title('WP Contestation Timeline')
ax2.set_xlabel('Election Year')
ax2.set_ylabel('')

# Chart 3: Paired bar chart per transition
ax3 = axes[1, 0]
bar_data = []
for trans in churn_df.transition.unique():
    sub = churn_df[churn_df.transition == trans]
    for contested in [True, False]:
        vals = sub[sub.wp_contested == contested]['churn_rate']
        if len(vals) > 0:
            bar_data.append({
                'transition': trans,
                'WP Contested': 'Yes' if contested else 'No',
                'mean_churn': vals.mean(),
                'ci': 1.96 * vals.std() / np.sqrt(len(vals)) if len(vals) > 1 else 0
            })
bar_df = pd.DataFrame(bar_data)
x = np.arange(len(churn_df.transition.unique()))
width = 0.35
wp_bars = bar_df[bar_df['WP Contested'] == 'Yes']
nonwp_bars = bar_df[bar_df['WP Contested'] == 'No']
ax3.bar(x - width/2, nonwp_bars['mean_churn'], width, yerr=nonwp_bars['ci'],
        label='Non-WP', color='#bdbdbd', capsize=4)
ax3.bar(x + width/2, wp_bars['mean_churn'], width, yerr=wp_bars['ci'],
        label='WP Contested', color='#08519c', capsize=4)
ax3.set_xticks(x)
ax3.set_xticklabels(churn_df.transition.unique(), rotation=15)
ax3.set_ylabel('Mean Churn Rate')
ax3.set_title('Mean Churn Rate by Transition')
ax3.legend()

# Chart 4: Scatter — WP count vs boundary change count
ax4 = axes[1, 1]
# For each cell, compute wp_count
def compute_wp_count(h):
    count = 0
    for i, year in enumerate(years):
        if h[i] in wp_contested[year]:
            count += 1
    return count

boundary_gdf['wp_count'] = boundary_gdf['h'].apply(compute_wp_count)
# Aggregate: mean boundary change count per wp_count level
scatter_data = boundary_gdf.groupby('wp_count').agg(
    mean_c=('c', 'mean'),
    count=('c', 'count'),
    std_c=('c', 'std')
).reset_index()
ax4.bar(scatter_data['wp_count'], scatter_data['mean_c'],
        color=['#f0f0f0', '#c6dbef', '#6baed6', '#2171b5', '#08519c', '#08306b'][:len(scatter_data)],
        edgecolor='#333', linewidth=0.5)
for _, row in scatter_data.iterrows():
    ax4.text(row.wp_count, row.mean_c + 0.05, f'n={int(row["count"])}',
             ha='center', va='bottom', fontsize=8)
ax4.set_xlabel('Times WP Contested (0–5)')
ax4.set_ylabel('Mean Boundary Changes')
ax4.set_title('WP Contestation vs Boundary Change Frequency')

plt.tight_layout()
plt.savefig('wp_contestation_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print('Charts saved to wp_contestation_charts.png')

Charts saved to wp_contestation_charts.png


In [7]:
# Cell 5: Prepare Map 2 data — WP contestation heatmap
# Add wp_count, wp_years, wp_details to each cell in boundary_changes_final.geojson

def build_wp_details(h):
    """Build WP details dict for a cell given its constituency history."""
    details = {}
    wp_years_list = []
    for i, year in enumerate(years):
        const_name = h[i]
        if const_name in wp_contested[year]:
            info = wp_results[year][const_name]
            details[str(year)] = {
                'pct': info['vote_pct'],
                'candidates': info['candidates'],
                'constituency': const_name
            }
            wp_years_list.append(year)
    return details, wp_years_list

wp_counts = []
wp_years_col = []
wp_details_col = []

for _, row in boundary_gdf.iterrows():
    h = row['h']
    details, wp_yrs = build_wp_details(h)
    wp_counts.append(len(wp_yrs))
    wp_years_col.append(wp_yrs)
    wp_details_col.append(json.dumps(details))

boundary_gdf['wp_count'] = wp_counts
boundary_gdf['wp_years'] = wp_years_col
boundary_gdf['wp_details'] = wp_details_col

print('WP count distribution across cells:')
print(boundary_gdf.wp_count.value_counts().sort_index())
print()
print('Spot check — cells with wp_count == 5:')
print(boundary_gdf[boundary_gdf.wp_count == 5][['n', 'h', 'c', 'wp_count']].head(10))
print()
print('Spot check — Aljunied cells:')
alj = boundary_gdf[boundary_gdf['h'].apply(lambda h: 'ALJUNIED' in h)]
print(f'  Aljunied cells: {len(alj)}, wp_count values: {alj.wp_count.unique()}')
print()
print('Spot check — Hougang cells:')
hg = boundary_gdf[boundary_gdf['h'].apply(lambda h: 'HOUGANG' in h)]
print(f'  Hougang cells: {len(hg)}, wp_count values: {hg.wp_count.unique()}')

WP count distribution across cells:
wp_count
0    797
1    326
2    227
3    166
4    137
5    109
Name: count, dtype: int64

Spot check — cells with wp_count == 5:
                     n                                                  h  c  \
2             ALJUNIED   [ALJUNIED, HOUGANG, ALJUNIED, HOUGANG, ALJUNIED]  4   
5             ALJUNIED  [ALJUNIED, PUNGGOL EAST, ALJUNIED, SENGKANG, A...  4   
50          EAST COAST  [EAST COAST, JOO CHIAT, EAST COAST, MARINE PAR...  4   
69             HOUGANG    [HOUGANG, ALJUNIED, HOUGANG, ALJUNIED, HOUGANG]  4   
180           TAMPINES  [ALJUNIED, EAST COAST, ALJUNIED, EAST COAST, T...  4   
184  TAMPINES CHANGKAT  [ALJUNIED, EAST COAST, ALJUNIED, EAST COAST, T...  4   
213           ALJUNIED  [ALJUNIED, ALJUNIED, PUNGGOL EAST, SENGKANG, A...  3   
223           ALJUNIED   [HOUGANG, ALJUNIED, ALJUNIED, HOUGANG, ALJUNIED]  3   
226           ALJUNIED   [HOUGANG, ALJUNIED, HOUGANG, ALJUNIED, ALJUNIED]  3   
227           ALJUNIED    [HOUGANG,

In [8]:
# Cell 6: Prepare Map 3 data — interaction score
# interaction_score = wp_count * c (interaction of WP contestation and boundary changes)

boundary_gdf['interaction_score'] = boundary_gdf['wp_count'] * boundary_gdf['c']

print('Interaction score distribution:')
print(boundary_gdf.interaction_score.value_counts().sort_index())
print()
print('High interaction_score cells (score >= 6):')
high_interaction = boundary_gdf[boundary_gdf.interaction_score >= 6][['n', 'h', 'c', 'wp_count', 'interaction_score']]
print(f'  Count: {len(high_interaction)}')
if len(high_interaction) > 0:
    print(high_interaction.head(20))
print()

# Bivariate categories for Map 3
def bivariate_cat(row):
    wp = row['wp_count'] > 0
    c = row['c']
    if not wp:
        return 'non-wp'
    elif c <= 1:
        return 'wp-low'
    elif c == 2:
        return 'wp-mid'
    else:  # c >= 3
        return 'wp-high'

boundary_gdf['bivar_cat'] = boundary_gdf.apply(bivariate_cat, axis=1)
print('Bivariate category distribution:')
print(boundary_gdf.bivar_cat.value_counts())

# Export: We won't export a separate file — we'll embed the data directly in index.html
# But let's build the lookup table we need for JS

# Build a per-cell wp_count + wp_details lookup keyed by feature index
wp_cell_data = []
for idx, row in boundary_gdf.iterrows():
    wp_cell_data.append({
        'wp': row['wp_count'],
        'gs': row['interaction_score'],
        'd': json.loads(row['wp_details']) if row['wp_details'] != '{}' else {}
    })

# Save as compact JSON for embedding
with open('../wp_cell_data.json', 'w') as f:
    json.dump(wp_cell_data, f, separators=(',', ':'))
    
import os
print(f'\nExported wp_cell_data.json: {os.path.getsize("../wp_cell_data.json") / 1024:.1f} KB')

Interaction score distribution:
interaction_score
0     803
1      36
2     142
3     147
4     138
5      23
6     156
8      79
9      64
10     46
12     80
15     31
16     11
20      6
Name: count, dtype: int64

High interaction_score cells (score >= 6):
  Count: 473
                   n                                                  h  c  \
0           ALJUNIED  [ALJUNIED, ANG MO KIO, ALJUNIED, ANG MO KIO, A...  4   
1           ALJUNIED  [ALJUNIED, BISHAN-TOA PAYOH, ALJUNIED, BISHAN-...  4   
2           ALJUNIED   [ALJUNIED, HOUGANG, ALJUNIED, HOUGANG, ALJUNIED]  4   
3           ALJUNIED  [ALJUNIED, MARINE PARADE, ALJUNIED, MARINE PAR...  4   
4           ALJUNIED  [ALJUNIED, PASIR RIS-PUNGGOL, ALJUNIED, ANG MO...  4   
5           ALJUNIED  [ALJUNIED, PUNGGOL EAST, ALJUNIED, SENGKANG, A...  4   
6           ALJUNIED  [PASIR RIS-PUNGGOL, ALJUNIED, PASIR RIS-PUNGGO...  4   
7           ALJUNIED  [PASIR RIS-PUNGGOL, ALJUNIED, PASIR RIS-PUNGGO...  4   
8           ALJUNIED  [PA

In [9]:
# Cell 7: Summary findings

print('='*60)
print('SUMMARY OF FINDINGS')
print('='*60)
print()
print('1. DESCRIPTIVE:')
print(f'   - {len(wp_churn)} constituency-transition pairs where WP contested')
print(f'   - {len(nonwp_churn)} constituency-transition pairs where WP did not contest')
print(f'   - Mean churn rate (WP): {np.mean(wp_churn)*100:.1f}%')
print(f'   - Mean churn rate (non-WP): {np.mean(nonwp_churn)*100:.1f}%')
print(f'   - Difference: {(np.mean(wp_churn) - np.mean(nonwp_churn))*100:.1f} percentage points')
print()
print('2. STATISTICAL TESTS:')
print(f'   - Pooled Mann-Whitney U: p={p_val:.4f}')
print(f'   - Permutation test: p={p_perm:.4f}')
print(f'   - Point-biserial r: {r_pb:.4f}, p={p_pb:.4f}')
print(f'   - Fisher exact: OR={odds_ratio:.3f}, p={p_fisher:.4f}')
print()
print('3. SPATIAL:')
wp_cells = boundary_gdf[boundary_gdf.wp_count > 0]
nonwp_cells = boundary_gdf[boundary_gdf.wp_count == 0]
print(f'   - Cells in WP-contested areas: {len(wp_cells)} ({len(wp_cells)/len(boundary_gdf)*100:.1f}%)')
print(f'   - Mean boundary changes (WP cells): {wp_cells.c.mean():.2f}')
print(f'   - Mean boundary changes (non-WP cells): {nonwp_cells.c.mean():.2f}')
print()
print('4. NOTABLE CASES:')
for const in ['ALJUNIED', 'HOUGANG', 'SENGKANG', 'EAST COAST']:
    cells = boundary_gdf[boundary_gdf.n == const]
    if len(cells) == 0:
        cells = boundary_gdf[boundary_gdf['h'].apply(lambda h: const in h)]
    print(f'   - {const}: {len(cells)} cells, mean c={cells.c.mean():.2f}, wp_count mode={cells.wp_count.mode().values[0] if len(cells) > 0 else "N/A"}')
print()
print('5. CONCLUSION:')
if p_val < 0.05 and observed_diff > 0:
    print('   WP-contested constituencies show SIGNIFICANTLY HIGHER boundary churn.')
elif p_val < 0.05 and observed_diff < 0:
    print('   WP-contested constituencies show SIGNIFICANTLY LOWER boundary churn.')
else:
    print('   No statistically significant difference in boundary churn between WP and non-WP constituencies.')
    print('   This is consistent with the null hypothesis: boundary changes are not targeted at WP-contested areas.')

SUMMARY OF FINDINGS

1. DESCRIPTIVE:


NameError: name 'wp_churn' is not defined

In [ ]:
# Cell 8: WP contestation vs boundary changes — constituency-level scatter plot
# Also exports JSON for the interactive chart in index.html
#
# Key decisions:
# - WP contestation: binary per election. For each 2025 constituency + election year,
#   find which OLD constituency held the majority of area. If WP contested it, count 1.
#   Sum across 5 elections → clean integer (0–5).
# - Boundary changes: binary per transition. For each 2025 constituency and each of the
#   4 transitions, compute what fraction of area changed constituency. If >10% of area
#   changed, that's a real boundary change (vs tiny edge adjustments). Sum → integer (0–4).
#   Threshold rationale: <10% churn = minor edge adjustments from new developments or
#   trivial realignment. >10% = substantive boundary redrawing that voters would notice.

import os

# --- Per-cell WP average vote share (for color) ---
def get_cell_wp_avg_vote(h):
    """Average WP vote share for this cell across elections where WP contested."""
    votes = []
    for i, year in enumerate(years):
        const_name = str(h[i])
        if const_name in wp_contested[year]:
            pct = wp_results[year].get(const_name, {}).get('vote_pct', 0)
            if pct:
                votes.append(pct)
    return np.mean(votes) if votes else 0

boundary_gdf['wp_avg_vote'] = boundary_gdf['h'].apply(get_cell_wp_avg_vote)

# --- Aggregate to 2025 constituency level ---
# h indices: 0=2006, 1=2011, 2=2015, 3=2020, 4=2025
transition_pairs = [(0, 1), (1, 2), (2, 3), (3, 4)]  # 4 transitions
CHURN_THRESHOLD = 0.10  # 10% of area must change for it to count as a boundary change

agg_rows = []
for const_name, group in boundary_gdf.groupby('n'):
    weights = group['area_m2'].values
    total_area = weights.sum()

    # === Integer WP contestation count (majority-area per election year) ===
    wp_count_int = 0
    wp_years_list = []
    wp_vote_pcts = []
    for i, year in enumerate(years):
        cell_consts = group['h'].apply(lambda h: str(h[i])).values
        const_areas = {}
        for j, c in enumerate(cell_consts):
            const_areas[c] = const_areas.get(c, 0) + weights[j]
        dominant = max(const_areas, key=const_areas.get)
        if dominant in wp_contested[year]:
            wp_count_int += 1
            wp_years_list.append(year)
            pct = wp_results[year].get(dominant, {}).get('vote_pct', 0)
            if pct:
                wp_vote_pcts.append(pct)

    # === Integer boundary change count (10% churn threshold per transition) ===
    boundary_change_int = 0
    for idx_t, idx_t1 in transition_pairs:
        consts_t = group['h'].apply(lambda h: str(h[idx_t])).values
        consts_t1 = group['h'].apply(lambda h: str(h[idx_t1])).values
        changed = np.array([is_genuine_change(o, n) for o, n in zip(consts_t, consts_t1)])
        churn = float(weights[changed].sum() / total_area) if total_area > 0 else 0
        if churn > CHURN_THRESHOLD:
            boundary_change_int += 1

    avg_wp_vote = np.mean(wp_vote_pcts) if wp_vote_pcts else 0

    agg_rows.append({
        'constituency': const_name,
        'boundary_changes': boundary_change_int,  # Clean integer 0-4
        'wp_elections': wp_count_int,  # Clean integer 0-5
        'wp_avg_vote': round(float(avg_wp_vote), 1),
        'wp_years': wp_years_list,
        'total_area': round(float(weights.sum()), 0),
        'n_cells': len(group),
    })

const_data = pd.DataFrame(agg_rows)

# --- Correlation and regression ---
r_val, p_val_corr = stats.pearsonr(const_data['wp_elections'], const_data['boundary_changes'])
slope, intercept, _, _, _ = stats.linregress(const_data['wp_elections'], const_data['boundary_changes'])
mean_y = const_data['boundary_changes'].mean()

print(f'Number of 2025 constituencies: {len(const_data)}')
print(f'WP elections range: {const_data.wp_elections.min()} to {const_data.wp_elections.max()}')
print(f'Boundary changes range: {const_data.boundary_changes.min()} to {const_data.boundary_changes.max()}')
print(f'Correlation: r={r_val:.3f}, p={p_val_corr:.3f}')
print(f'Regression: slope={slope:.4f}, intercept={intercept:.4f}')
print(f'Mean boundary changes: {mean_y:.3f}')
print(f'Churn threshold: {CHURN_THRESHOLD*100:.0f}%')
print()
print('WP elections distribution:')
print(const_data.wp_elections.value_counts().sort_index())
print()
print('Boundary changes distribution:')
print(const_data.boundary_changes.value_counts().sort_index())
print()
print(const_data.sort_values('wp_elections', ascending=False)[
    ['constituency', 'wp_elections', 'wp_years', 'wp_avg_vote', 'boundary_changes']
].to_string(index=False))

# --- Export JSON for index.html interactive chart ---
chart_json = {
    'points': [
        {'n': r['constituency'], 'bc': int(r['boundary_changes']), 'we': int(r['wp_elections']),
         'wv': r['wp_avg_vote'], 'a': r['total_area']}
        for _, r in const_data.iterrows()
    ],
    'r': round(r_val, 2),
    'p': round(p_val_corr, 2),
    'slope': round(slope, 4),
    'intercept': round(intercept, 4),
    'meanY': round(mean_y, 3),
}
with open('../wp_scatter_chart_data.json', 'w') as f:
    json.dump(chart_json, f, separators=(',', ':'))
print(f'\nExported wp_scatter_chart_data.json: {os.path.getsize("../wp_scatter_chart_data.json") / 1024:.1f} KB')

# --- Create scatter plot (matplotlib version) ---
fig, ax = plt.subplots(figsize=(12, 9))

# Add jitter for readability (both axes are integers now)
np.random.seed(42)
jitter_x = np.random.uniform(-0.15, 0.15, len(const_data))
jitter_y = np.random.uniform(-0.12, 0.12, len(const_data))

size_factor = const_data['total_area'] / const_data['total_area'].median()
sizes = np.clip(size_factor * 120, 40, 500)

scatter = ax.scatter(
    const_data['wp_elections'] + jitter_x,
    const_data['boundary_changes'] + jitter_y,
    c=const_data['wp_avg_vote'],
    s=sizes,
    cmap='YlOrRd',
    alpha=0.8,
    edgecolors='#333',
    linewidth=0.5,
    zorder=5,
    vmin=0,
    vmax=55,
)

cbar = plt.colorbar(scatter, ax=ax, pad=0.02)
cbar.set_label('Avg WP Vote %\n(when contested)', fontsize=11)

# Regression line
x_line = np.linspace(-0.5, 5.5, 100)
ax.plot(x_line, slope * x_line + intercept, 'k--', linewidth=2, alpha=0.7, zorder=4)

# Mean horizontal line
ax.axhline(mean_y, color='blue', linestyle=':', alpha=0.5, linewidth=1)

# Labels for key WP constituencies
label_configs = {
    'HOUGANG': {'offset': (12, 8), 'ha': 'left'},
    'ALJUNIED': {'offset': (12, -10), 'ha': 'left'},
    'SENGKANG': {'offset': (10, 8), 'ha': 'left'},
    'EAST COAST': {'offset': (10, 8), 'ha': 'left'},
    'MARINE PARADE-BRADDELL HEIGHTS': {'offset': (-10, 8), 'ha': 'right', 'label': 'MARINE PARADE'},
    'PASIR RIS-CHANGI': {'offset': (10, -12), 'ha': 'left'},
}
for i, (idx, row) in enumerate(const_data.iterrows()):
    if row['constituency'] in label_configs or row['wp_elections'] >= 4:
        config = label_configs.get(row['constituency'], {'offset': (10, 5), 'ha': 'left'})
        label = config.get('label', row['constituency'])
        ax.annotate(
            label,
            (row['wp_elections'] + jitter_x[i], row['boundary_changes'] + jitter_y[i]),
            textcoords="offset points",
            xytext=config['offset'],
            fontsize=9,
            fontweight='bold',
            ha=config['ha'],
            zorder=10,
        )

ax.set_xlabel('Elections WP contested (out of 5, 2006\u20132025)', fontsize=12)
ax.set_ylabel('Times boundary changed (out of 4 transitions)', fontsize=12)
ax.set_xticks([0, 1, 2, 3, 4, 5])
ax.set_yticks([0, 1, 2, 3, 4])
corr_label = 'No Correlation' if p_val_corr > 0.05 else (
    'Weak Correlation' if abs(r_val) < 0.3 else 'Correlation Found')
ax.set_title(
    f'WP Contestation vs Boundary Changes: {corr_label}\n(r = {r_val:.2f}, p = {p_val_corr:.2f})',
    fontsize=14, fontweight='bold'
)

ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../output/images/wp_contestation_vs_boundary_changes.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to ../output/images/wp_contestation_vs_boundary_changes.png')

Number of 2025 constituencies: 33
WP elections range: 0 to 5
Boundary changes range: 0 to 3
Correlation: r=-0.123, p=0.494
Regression: slope=-0.0676, intercept=1.5586
Mean boundary changes: 1.485
Churn threshold: 10%

WP elections distribution:
wp_elections
0    18
1     5
2     6
4     1
5     3
Name: count, dtype: int64

Boundary changes distribution:
boundary_changes
0     4
1    13
2    12
3     4
Name: count, dtype: int64

                  constituency  wp_elections                       wp_years  wp_avg_vote  boundary_changes
                      ALJUNIED             5 [2006, 2011, 2015, 2020, 2025]         53.9                 0
                    EAST COAST             5 [2006, 2011, 2015, 2020, 2025]         41.7                 2
                       HOUGANG             5 [2006, 2011, 2015, 2020, 2025]         61.7                 0
              PASIR RIS-CHANGI             4       [2006, 2011, 2015, 2020]         41.8                 1
MARINE PARADE-BRADDELL HEIGHTS   

Saved to ../output/images/wp_contestation_vs_boundary_changes.png


In [ ]:
# Cell 9: Cell-level bubble chart — WP contestation vs boundary changes
# Each of the 1,762 cells (~50m parcels) has clean integers:
#   x = wp_count (0–5): how many of 5 elections this cell was in a WP-contested constituency
#   y = c (0–4): how many times this cell changed constituency name
# No aggregation, no thresholds. This is the most granular and honest view.
# Bubbles are placed at each (x, y) grid position, sized by cell count.

import os

# wp_count was already computed in Cell 6; recompute here for safety
def compute_wp_count(h):
    return sum(1 for i, year in enumerate(years) if str(h[i]) in wp_contested[year])

boundary_gdf['wp_count'] = boundary_gdf['h'].apply(compute_wp_count)

# Per-cell average WP vote share (for bubble colour)
def get_cell_wp_avg_vote(h):
    votes = []
    for i, year in enumerate(years):
        const_name = str(h[i])
        if const_name in wp_contested[year]:
            pct = wp_results[year].get(const_name, {}).get('vote_pct', 0)
            if pct:
                votes.append(pct)
    return np.mean(votes) if votes else 0

boundary_gdf['wp_avg_vote'] = boundary_gdf['h'].apply(get_cell_wp_avg_vote)

# --- Cross-tabulation ---
ct = pd.crosstab(boundary_gdf.wp_count, boundary_gdf.c, margins=True)
print('Cross-tab (wp_count rows × boundary changes cols):')
print(ct)
print()

# --- Correlation ---
r_val, p_val_corr = stats.pearsonr(boundary_gdf['wp_count'], boundary_gdf['c'])
slope, intercept, _, _, _ = stats.linregress(boundary_gdf['wp_count'], boundary_gdf['c'])
mean_y = boundary_gdf['c'].mean()
print(f'Pearson r = {r_val:.3f}, p = {p_val_corr:.4f}')
print(f'Slope = {slope:.4f}, intercept = {intercept:.4f}')
print(f'Mean boundary changes = {mean_y:.3f}')
print(f'Total cells = {len(boundary_gdf)}')

# --- Aggregate to (wp_count, c) bubbles for chart ---
bubble_df = boundary_gdf.groupby(['wp_count', 'c']).agg(
    count=('n', 'count'),
    avg_vote=('wp_avg_vote', 'mean'),
    constituencies=('n', lambda x: sorted(x.unique())),
).reset_index()

print()
print('Bubble data:')
for _, row in bubble_df.iterrows():
    print(f'  wp={row.wp_count}, c={row.c}: {row["count"]:4d} cells, '
          f'avg WP vote={row.avg_vote:5.1f}%, '
          f'in {len(row.constituencies)} constituencies')

# --- Export JSON for index.html ---
chart_json = {
    'bubbles': [
        {'x': int(r.wp_count), 'y': int(r.c), 'count': int(r['count']),
         'wv': round(r.avg_vote, 1),
         'consts': r.constituencies}
        for _, r in bubble_df.iterrows()
    ],
    'r': round(r_val, 3),
    'p': round(p_val_corr, 4),
    'slope': round(slope, 4),
    'intercept': round(intercept, 4),
    'meanY': round(mean_y, 3),
    'total': len(boundary_gdf),
}
with open('../wp_scatter_chart_data.json', 'w') as f:
    json.dump(chart_json, f, separators=(',', ':'))
print(f'\nExported wp_scatter_chart_data.json: {os.path.getsize("../wp_scatter_chart_data.json") / 1024:.1f} KB')

# --- Bubble chart (matplotlib) ---
fig, ax = plt.subplots(figsize=(10, 8))

# Bubble sizes: sqrt-scaled, largest bubble for 290 cells
max_count = bubble_df['count'].max()
bubble_df['size'] = (bubble_df['count'] / max_count) ** 0.5 * 2000

# Colour: grey for wp=0, blue scale for wp>0 based on avg WP vote
def bubble_color(row):
    if row.wp_count == 0:
        return '#d5d5d2'
    wv = row.avg_vote
    if wv < 35: return '#c6dbef'
    if wv < 40: return '#9ecae1'
    if wv < 45: return '#6baed6'
    if wv < 50: return '#3182bd'
    if wv < 55: return '#08519c'
    return '#08306b'

colors = bubble_df.apply(bubble_color, axis=1)

ax.scatter(
    bubble_df['wp_count'], bubble_df['c'],
    s=bubble_df['size'], c=colors,
    alpha=0.8, edgecolors='#333', linewidth=0.8, zorder=5
)

# Count labels inside bubbles
for _, row in bubble_df.iterrows():
    if row['count'] >= 5:
        ax.annotate(
            str(int(row['count'])),
            (row.wp_count, row.c),
            ha='center', va='center',
            fontsize=8 if row['count'] < 50 else 9,
            fontweight='bold',
            color='#fff' if row.wp_count >= 3 or row['count'] > 200 else '#333',
            zorder=10,
        )

# Regression line
x_line = np.linspace(-0.5, 5.5, 100)
ax.plot(x_line, slope * x_line + intercept, 'k--', linewidth=1.5, alpha=0.5, zorder=4)

# Mean line
ax.axhline(mean_y, color='#08519c', linestyle=':', alpha=0.4, linewidth=1)
ax.text(5.6, mean_y, 'avg', fontsize=8, color='#08519c', va='center', alpha=0.6)

ax.set_xlabel('Times in WP-contested constituency (out of 5 elections)', fontsize=12)
ax.set_ylabel('Times boundary changed (out of 4 transitions)', fontsize=12)
ax.set_xticks([0, 1, 2, 3, 4, 5])
ax.set_yticks([0, 1, 2, 3, 4])
ax.set_xlim(-0.5, 5.8)
ax.set_ylim(-0.5, 4.5)

corr_label = 'No Correlation' if p_val_corr > 0.05 else (
    'Weak Correlation' if abs(r_val) < 0.3 else 'Correlation Found')
ax.set_title(
    f'WP Contestation vs Boundary Changes (cell-level): {corr_label}\n'
    f'(n = {len(boundary_gdf):,} cells, r = {r_val:.3f}, p = {p_val_corr:.4f})',
    fontsize=13, fontweight='bold'
)
ax.text(0.02, 0.02, 'Bubble size = number of cells. Grey = never WP-contested. Blue = WP vote share.',
        transform=ax.transAxes, fontsize=8, color='#666', va='bottom')

ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig('../output/images/wp_contestation_vs_boundary_changes_cells.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to ../output/images/wp_contestation_vs_boundary_changes_cells.png')

Cross-tab (wp_count rows × boundary changes cols):
c          0    1    2    3    4   All
wp_count                              
0          7  123  266  290  111   797
1          2   35  113  129   47   326
2          0   21   74  101   31   227
3          0   20   55   67   24   166
4          0   14   53   59   11   137
5          3   23   46   31    6   109
All       12  236  607  677  230  1762

Pearson r = -0.035, p = 0.1404
Slope = -0.0200, intercept = 2.5247
Mean boundary changes = 2.498
Total cells = 1762

Bubble data:
  wp=0, c=0:    7 cells, avg WP vote=  0.0%, in 7 constituencies
  wp=0, c=1:  123 cells, avg WP vote=  0.0%, in 23 constituencies
  wp=0, c=2:  266 cells, avg WP vote=  0.0%, in 24 constituencies
  wp=0, c=3:  290 cells, avg WP vote=  0.0%, in 23 constituencies
  wp=0, c=4:  111 cells, avg WP vote=  0.0%, in 18 constituencies
  wp=1, c=0:    2 cells, avg WP vote= 40.6%, in 2 constituencies
  wp=1, c=1:   35 cells, avg WP vote= 40.8%, in 17 constituencies
  wp=1,

Saved to ../output/images/wp_contestation_vs_boundary_changes_cells.png


In [ ]:
# Cell 10: Does WP expansion (not strongholds) trigger more boundary changes?
#
# Hypothesis: WP strongholds (Hougang, Aljunied) are left alone, but when WP
# tries to expand into new territory, boundaries get redrawn more aggressively.
#
# Tests:
# 1. Stronghold vs Expansion vs Non-WP group comparison
# 2. Temporal: does WP contesting in year t trigger redrawing in t→t+1?
# 3. First-time WP expansion: new contest → boundary change?
# 4. WP strong showing (>45%) → boundary change in next cycle?

# Rebuild constituency-level data with 10% churn threshold
transition_pairs_idx = [(0, 1), (1, 2), (2, 3), (3, 4)]
CHURN_THRESHOLD = 0.10

agg_rows_11 = []
for const_name, group in boundary_gdf.groupby('n'):
    weights = group['area_m2'].values
    total_area = weights.sum()

    wp_count_int = 0
    wp_years_list = []
    wp_vote_pcts = []
    for i, year in enumerate(years):
        cell_consts = group['h'].apply(lambda h: str(h[i])).values
        const_areas = {}
        for j, c in enumerate(cell_consts):
            const_areas[c] = const_areas.get(c, 0) + weights[j]
        dominant = max(const_areas, key=const_areas.get)
        if dominant in wp_contested[year]:
            wp_count_int += 1
            wp_years_list.append(year)
            pct = wp_results[year].get(dominant, {}).get('vote_pct', 0)
            if pct: wp_vote_pcts.append(pct)

    bc_int = 0
    bc_transitions = []
    for idx_t, idx_t1 in transition_pairs_idx:
        consts_t = group['h'].apply(lambda h: str(h[idx_t])).values
        consts_t1 = group['h'].apply(lambda h: str(h[idx_t1])).values
        changed = np.array([is_genuine_change(o, n) for o, n in zip(consts_t, consts_t1)])
        churn = float(weights[changed].sum() / total_area) if total_area > 0 else 0
        if churn > CHURN_THRESHOLD:
            bc_int += 1
            bc_transitions.append(f'{years[idx_t]}-{years[idx_t1]}')

    agg_rows_11.append({
        'constituency': const_name,
        'bc': bc_int,
        'we': wp_count_int,
        'wp_avg_vote': round(float(np.mean(wp_vote_pcts)) if wp_vote_pcts else 0, 1),
        'wp_years': wp_years_list,
        'bc_transitions': bc_transitions,
    })

df11 = pd.DataFrame(agg_rows_11)

# ═══════════════════════════════════════════════════════════════════
# TEST 1: Stronghold vs Expansion vs Non-WP
# ═══════════════════════════════════════════════════════════════════
print('='*70)
print('TEST 1: Strongholds vs Expansion zones vs Non-WP')
print('='*70)

def categorise(we):
    if we == 0: return 'Non-WP (0)'
    if we <= 3: return 'Expansion (1-3)'
    return 'Stronghold (4-5)'

df11['category'] = df11['we'].apply(categorise)

for cat in ['Non-WP (0)', 'Expansion (1-3)', 'Stronghold (4-5)']:
    sub = df11[df11.category == cat]
    print(f'\n  {cat}: n={len(sub)}, mean bc={sub.bc.mean():.2f}, median={sub.bc.median():.1f}')
    for _, r in sub.sort_values('bc', ascending=False).iterrows():
        print(f'    {r.constituency:35s}  bc={r.bc}  we={r.we}  wp_vote={r.wp_avg_vote}%')

exp = df11[df11.category == 'Expansion (1-3)']['bc'].values
non = df11[df11.category == 'Non-WP (0)']['bc'].values
strong = df11[df11.category == 'Stronghold (4-5)']['bc'].values

u1, p1 = stats.mannwhitneyu(exp, non, alternative='two-sided')
u2, p2 = stats.mannwhitneyu(strong, non, alternative='two-sided')
u3, p3 = stats.mannwhitneyu(exp, strong, alternative='two-sided')
h_stat, p_kw = stats.kruskal(non, exp, strong)

print(f'\n  Mann-Whitney U tests:')
print(f'    Expansion vs Non-WP:    U={u1:.1f}, p={p1:.4f}')
print(f'    Stronghold vs Non-WP:   U={u2:.1f}, p={p2:.4f}')
print(f'    Expansion vs Stronghold: U={u3:.1f}, p={p3:.4f}')
print(f'  Kruskal-Wallis (3 groups): H={h_stat:.2f}, p={p_kw:.4f}')

# ═══════════════════════════════════════════════════════════════════
# TEST 2: Temporal — WP contests at year t → boundary change in t→t+1?
# ═══════════════════════════════════════════════════════════════════
print()
print('='*70)
print('TEST 2: Temporal — WP contests at time t → redrawn in t→t+1?')
print('='*70)

temporal_rows = []
for _, row in df11.iterrows():
    for i, (idx_t, idx_t1) in enumerate(transition_pairs_idx):
        year_t = years[idx_t]
        wp_before = year_t in row['wp_years']
        bc_after = f'{years[idx_t]}-{years[idx_t1]}' in row['bc_transitions']
        temporal_rows.append({
            'constituency': row['constituency'],
            'transition': f'{years[idx_t]}-{years[idx_t1]}',
            'wp_contested_before': wp_before,
            'boundary_changed': bc_after,
        })

tdf = pd.DataFrame(temporal_rows)
ct = pd.crosstab(tdf.wp_contested_before, tdf.boundary_changed, margins=True)
print('\n  Contingency table (WP contested before → boundary changed after):')
print(ct)

or_val, p_fisher = stats.fisher_exact(pd.crosstab(tdf.wp_contested_before, tdf.boundary_changed))
wp_rate = tdf[tdf.wp_contested_before]['boundary_changed'].mean()
non_rate = tdf[~tdf.wp_contested_before]['boundary_changed'].mean()
print(f'\n  Fisher exact: OR={or_val:.3f}, p={p_fisher:.4f}')
print(f'  WP-contested → redrawn: {wp_rate*100:.1f}%')
print(f'  Not WP-contested → redrawn: {non_rate*100:.1f}%')

# ═══════════════════════════════════════════════════════════════════
# TEST 3: First-time WP expansion — new WP contest → redrawn?
# ═══════════════════════════════════════════════════════════════════
print()
print('='*70)
print('TEST 3: First-time WP expansion — new contest → redrawn?')
print('='*70)

ft_rows = []
for _, row in df11.iterrows():
    wp_yrs = sorted(row['wp_years'])
    if not wp_yrs: continue
    first_yr = wp_yrs[0]
    for i, (idx_t, idx_t1) in enumerate(transition_pairs_idx):
        if years[idx_t] == first_yr:
            bc_after = f'{years[idx_t]}-{years[idx_t1]}' in row['bc_transitions']
            ft_rows.append({
                'constituency': row['constituency'],
                'first_wp_year': first_yr,
                'transition': f'{years[idx_t]}-{years[idx_t1]}',
                'redrawn_after': bc_after,
            })
            break

ftdf = pd.DataFrame(ft_rows)
base_rate = tdf.boundary_changed.mean()
print(f'\n  First-time WP expansions: {len(ftdf)}')
print(f'  Redrawn after first contest: {ftdf.redrawn_after.sum()} of {len(ftdf)} ({ftdf.redrawn_after.mean()*100:.1f}%)')
print(f'  Base rate (all transitions): {base_rate*100:.1f}%')
print()
for _, r in ftdf.iterrows():
    tag = 'REDRAWN' if r.redrawn_after else 'stable'
    print(f'    {r.constituency:35s}  first WP {r.first_wp_year} → {r.transition}: {tag}')

# ═══════════════════════════════════════════════════════════════════
# TEST 4: WP strong showing (>45%) → boundary change?
# ═══════════════════════════════════════════════════════════════════
print()
print('='*70)
print('TEST 4: WP strong showing (>45%) → redrawn in next cycle?')
print('='*70)

ss_rows = []
for _, row in df11.iterrows():
    for i, (idx_t, idx_t1) in enumerate(transition_pairs_idx):
        year_t = years[idx_t]
        if year_t not in row['wp_years']: continue
        group = boundary_gdf[boundary_gdf.n == row['constituency']]
        weights = group['area_m2'].values
        cell_consts = group['h'].apply(lambda h: str(h[idx_t])).values
        const_areas = {}
        for j, c in enumerate(cell_consts):
            const_areas[c] = const_areas.get(c, 0) + weights[j]
        dominant = max(const_areas, key=const_areas.get)
        pct = wp_results.get(year_t, {}).get(dominant, {}).get('vote_pct', 0)
        bc_after = f'{years[idx_t]}-{years[idx_t1]}' in row['bc_transitions']
        ss_rows.append({
            'constituency': row['constituency'],
            'year': year_t,
            'wp_vote': pct,
            'strong': pct > 45 if pct else False,
            'redrawn_after': bc_after,
        })

ssdf = pd.DataFrame(ss_rows)
strong_rate = ssdf[ssdf.strong]['redrawn_after'].mean()
weak_rate = ssdf[~ssdf.strong]['redrawn_after'].mean()
print(f'\n  WP contests with >45%: {ssdf.strong.sum()}')
print(f'  Redrawn after strong showing (>45%): {strong_rate*100:.1f}%')
print(f'  Redrawn after weaker showing (<=45%): {weak_rate*100:.1f}%')
ct2 = pd.crosstab(ssdf.strong, ssdf.redrawn_after)
if ct2.shape == (2, 2):
    or2, p2 = stats.fisher_exact(ct2)
    print(f'  Fisher exact: OR={or2:.3f}, p={p2:.4f}')

# ═══════════════════════════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════════════════════════
print()
print('='*70)
print('SUMMARY: Does WP expansion trigger more boundary changes?')
print('='*70)
print(f'''
  Expansion zones (WP 1-3 elections) have the highest boundary change
  rate (mean {df11[df11.category=="Expansion (1-3)"].bc.mean():.2f}), compared to non-WP areas ({df11[df11.category=="Non-WP (0)"].bc.mean():.2f})
  and strongholds ({df11[df11.category=="Stronghold (4-5)"].bc.mean():.2f}). The Kruskal-Wallis test across all three
  groups gives p={p_kw:.4f} — borderline but not significant at p<0.05.

  Expansion vs Stronghold is the closest to significance (p={p3:.4f}),
  suggesting a possible pattern where WP's peripheral contests see
  more boundary volatility than its established seats.

  However, temporal analysis finds NO evidence of reactive redrawing:
  - WP contesting does not predict next-cycle boundary changes (p={p_fisher:.4f})
  - First-time WP expansion: {ftdf.redrawn_after.mean()*100:.1f}% redrawn vs {base_rate*100:.1f}% base rate
  - Strong WP showing (>45%) actually has FEWER subsequent changes
    ({strong_rate*100:.1f}%) than weaker showings ({weak_rate*100:.1f}%), OR={or2:.3f}, p={p2:.4f}

  Conclusion: the pattern in the scatter plot (expansion zones higher
  than strongholds) is suggestive but not statistically significant,
  and the temporal tests do not support a causal story of reactive
  gerrymandering.
''')

TEST 1: Strongholds vs Expansion zones vs Non-WP

  Non-WP (0): n=18, mean bc=1.39, median=1.0
    JURONG EAST-BUKIT BATOK              bc=3  we=0  wp_vote=0.0%
    BISHAN-TOA PAYOH                     bc=2  we=0  wp_vote=0.0%
    TANJONG PAGAR                        bc=2  we=0  wp_vote=0.0%
    SEMBAWANG                            bc=2  we=0  wp_vote=0.0%
    JURONG CENTRAL                       bc=2  we=0  wp_vote=0.0%
    MARSILING-YEW TEE                    bc=2  we=0  wp_vote=0.0%
    RADIN MAS                            bc=2  we=0  wp_vote=0.0%
    BUKIT GOMBAK                         bc=2  we=0  wp_vote=0.0%
    POTONG PASIR                         bc=1  we=0  wp_vote=0.0%
    SEMBAWANG WEST                       bc=1  we=0  wp_vote=0.0%
    QUEENSTOWN                           bc=1  we=0  wp_vote=0.0%
    MOUNTBATTEN                          bc=1  we=0  wp_vote=0.0%
    PIONEER                              bc=1  we=0  wp_vote=0.0%
    MARYMOUNT                            bc=1  

In [ ]:
# Cell 11: Interaction Score (interaction_score) — Statistical tests & choropleth
#
# interaction_score = wp_count × c (boundary changes)
# Tests whether this multiplicative interaction is significantly
# different from what chance would produce.

from scipy.stats import spearmanr, pearsonr, chi2_contingency

# ── Recompute interaction_score for safety ──
boundary_gdf['interaction_score'] = boundary_gdf['wp_count'] * boundary_gdf['c']

print('='*70)
print('INTERACTION SCORE: interaction_score = wp_count × boundary_changes')
print('='*70)
print()

# ── Descriptive stats ──
print('DESCRIPTIVE STATISTICS')
print('-'*50)
print(f'  N cells: {len(boundary_gdf)}')
print(f'  Mean:   {boundary_gdf.interaction_score.mean():.3f}')
print(f'  Median: {boundary_gdf.interaction_score.median():.1f}')
print(f'  Std:    {boundary_gdf.interaction_score.std():.3f}')
print(f'  Min:    {boundary_gdf.interaction_score.min()}')
print(f'  Max:    {boundary_gdf.interaction_score.max()}')
print()
print('Score distribution:')
print(boundary_gdf.interaction_score.value_counts().sort_index())
print()

# ── Test 1: Spearman rank correlation ──
print('='*70)
print('TEST 1: Spearman rank correlation (wp_count vs c)')
print('='*70)
rho, p_spearman = spearmanr(boundary_gdf['wp_count'], boundary_gdf['c'])
print(f'  rho = {rho:.4f}, p = {p_spearman:.6f}')
if p_spearman < 0.05:
    print(f'  → Significant at p<0.05: there IS a monotonic association.')
else:
    print(f'  → Not significant at p<0.05: no evidence of monotonic association.')
print()

# ── Test 2: Permutation test ──
print('='*70)
print('TEST 2: Permutation test (10,000 shuffles)')
print('='*70)
observed_interaction = (boundary_gdf['wp_count'] * boundary_gdf['c']).mean()
n_perm = 10000
rng = np.random.default_rng(42)
wp_vals = boundary_gdf['wp_count'].values
c_vals = boundary_gdf['c'].values
perm_means = np.empty(n_perm)
for i in range(n_perm):
    shuffled_wp = rng.permutation(wp_vals)
    perm_means[i] = np.mean(shuffled_wp * c_vals)
p_perm_interaction = np.mean(perm_means >= observed_interaction)
p_perm_twosided = np.mean(np.abs(perm_means - perm_means.mean()) >= np.abs(observed_interaction - perm_means.mean()))
print(f'  Observed mean(wp_count × c): {observed_interaction:.4f}')
print(f'  Permuted mean (avg):         {perm_means.mean():.4f}')
print(f'  Permuted mean (std):         {perm_means.std():.4f}')
print(f'  One-sided p (observed >= permuted): {p_perm_interaction:.4f}')
print(f'  Two-sided p: {p_perm_twosided:.4f}')
if p_perm_twosided < 0.05:
    print(f'  → Significant: observed interaction is unlikely under random assignment.')
else:
    print(f'  → Not significant: observed interaction is consistent with chance.')
print()

# ── Test 3: Chi-squared test on cross-tabulation ──
print('='*70)
print('TEST 3: Chi-squared test (wp_count × c cross-tabulation)')
print('='*70)
ct = pd.crosstab(boundary_gdf['wp_count'], boundary_gdf['c'])
print('Cross-tabulation:')
print(ct)
print()
chi2, p_chi2, dof, expected = chi2_contingency(ct)
print(f'  Chi-squared: {chi2:.2f}')
print(f'  Degrees of freedom: {dof}')
print(f'  p-value: {p_chi2:.6f}')
if p_chi2 < 0.05:
    print(f'  → Significant: wp_count and c are NOT independent.')
else:
    print(f'  → Not significant: no evidence that wp_count and c are dependent.')
print()

# ── Test 4: Pearson correlation ──
print('='*70)
print('TEST 4: Pearson correlation (wp_count vs c)')
print('='*70)
r_pearson, p_pearson = pearsonr(boundary_gdf['wp_count'], boundary_gdf['c'])
print(f'  r = {r_pearson:.4f}, p = {p_pearson:.6f}')
if p_pearson < 0.05:
    print(f'  → Significant linear correlation.')
else:
    print(f'  → No significant linear correlation.')
print()

# ── Choropleth of interaction_score ──
print('='*70)
print('CHOROPLETH: Interaction Score')
print('='*70)

from matplotlib.colors import ListedColormap, BoundaryNorm

fig, ax = plt.subplots(1, 1, figsize=(12, 10))

# Define bins matching the JS getInteractionColor
bounds = [0, 0.5, 3.5, 6.5, 10.5, 20]
colors_list = ['#f0f0f0', '#fdd49e', '#fc8d59', '#e34a33', '#8c1a0f']
cmap = ListedColormap(colors_list)
norm = BoundaryNorm(bounds, cmap.N)

boundary_gdf.plot(
    column='interaction_score',
    cmap=cmap,
    norm=norm,
    linewidth=0.2,
    edgecolor='#666',
    ax=ax,
    legend=False,
)

# Manual legend
import matplotlib.patches as mpatches
legend_labels = ['0 (No overlap)', '1–3 (Low)', '4–6 (Medium)', '8–10 (High)', '12+ (Very high)']
patches = [mpatches.Patch(facecolor=c, edgecolor='#666', label=l)
           for c, l in zip(colors_list, legend_labels)]
ax.legend(handles=patches, loc='lower left', fontsize=9, title='Interaction Score',
          title_fontsize=10, framealpha=0.9)

ax.set_title('Interaction Score: WP Contestation × Boundary Changes', fontsize=14, fontweight='bold')
ax.set_axis_off()
plt.tight_layout()
plt.savefig('../output/images/interaction_score_choropleth.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to ../output/images/interaction_score_choropleth.png')

# ── Summary ──
print()
print('='*70)
print('SUMMARY: Interaction Score Statistical Tests')
print('='*70)
print(f'''
  Spearman rho = {rho:.4f}, p = {p_spearman:.6f}
  Permutation test (two-sided): p = {p_perm_twosided:.4f}
  Chi-squared: χ² = {chi2:.2f}, p = {p_chi2:.6f}
  Pearson r = {r_pearson:.4f}, p = {p_pearson:.6f}

  {'All tests significant at p<0.05.' if all(p < 0.05 for p in [p_spearman, p_perm_twosided, p_chi2, p_pearson]) else 'Not all tests reach significance at p<0.05.'}
  
  High-score cells (score >= 8): {len(boundary_gdf[boundary_gdf.interaction_score >= 8])} of {len(boundary_gdf)}
  These are areas that were BOTH frequently WP-contested AND frequently redrawn.
''')

INTERACTION SCORE: interaction_score = wp_count × boundary_changes

DESCRIPTIVE STATISTICS
--------------------------------------------------
  N cells: 1762
  Mean:   3.310
  Median: 2.0
  Std:    4.162
  Min:    0
  Max:    20

Score distribution:
interaction_score
0     802
1      35
2     134
3     149
4     135
5      23
6     156
8      84
9      67
10     46
12     83
15     31
16     11
20      6
Name: count, dtype: int64

TEST 1: Spearman rank correlation (wp_count vs c)
  rho = -0.0079, p = 0.741553
  → Not significant at p<0.05: no evidence of monotonic association.

TEST 2: Permutation test (10,000 shuffles)


  Observed mean(wp_count × c): 3.3104
  Permuted mean (avg):         3.3612
  Permuted mean (std):         0.0346
  One-sided p (observed >= permuted): 0.9289
  Two-sided p: 0.1449
  → Not significant: observed interaction is consistent with chance.

TEST 3: Chi-squared test (wp_count × c cross-tabulation)
Cross-tabulation:
c         0    1    2    3    4
wp_count                       
0         7  123  266  290  111
1         2   35  113  129   47
2         0   21   74  101   31
3         0   20   55   67   24
4         0   14   53   59   11
5         3   23   46   31    6

  Chi-squared: 42.90
  Degrees of freedom: 20
  p-value: 0.002108
  → Significant: wp_count and c are NOT independent.

TEST 4: Pearson correlation (wp_count vs c)
  r = -0.0351, p = 0.140448
  → No significant linear correlation.

CHOROPLETH: Interaction Score


Saved to ../output/images/interaction_score_choropleth.png

SUMMARY: Interaction Score Statistical Tests

  Spearman rho = -0.0079, p = 0.741553
  Permutation test (two-sided): p = 0.1449
  Chi-squared: χ² = 42.90, p = 0.002108
  Pearson r = -0.0351, p = 0.140448

  Not all tests reach significance at p<0.05.
  
  High-score cells (score >= 8): 328 of 1762
  These are areas that were BOTH frequently WP-contested AND frequently redrawn.

